# Maximun Independent Set y Quantum Monte Carlo

## Motivación

Con el fin de asentar todos los conocimientos obtenidos hasta el momento, se busca implementar una forma de solucionar el Maximum Independent Set (MIS) haciendo uso del paquete Bloqade para sistemas del orden de las centenas de qbits. Para ello, se utiliza QMC para hacer el cálculo numérico mediante BloqadeQMC.

## Preparación

Comenzamos importando los paquetes necesarios para ello:

In [2]:
using Bloqade, Bloqade.CairoMakie #Bloqade y el módulo para gráficas y eso.
using Yao: mat, ArrayReg #Par de funciones para simplificar la comparación de ED y QMC
using LinearAlgebra #Cálculos para operaciones matriciales
using Measurements #Cálculos de propagación de errores
using Measurements: value, uncertainty
using Statistics #Funciones estadísticas
using BloqadeQMC #Módulo para Quantum Monte Carlo
using Random #Generación de números pseudoaleatorios y semillas
using BinningAnalysis
using GenericTensorNetworks #Resolver MIS
using Optim #Optimización
using Graphs #Gráficas.

El primer paso es generar la red a estudiar, sea una red 11 x 11 al que le quitamos un 20% de átomos. Definimos una separación entre átomos de 4.5 μm y un radio de bloqueo de 7.5 μm, de manera que se produce bloqueo Rydberg a primeros y segundos vecinos.

In [3]:
Random.seed!(13)
atomos = generate_sites(SquareLattice(),11,11, scale = 4.5) |> random_dropout(0.2); #Generamos la red.
n_sites = length(atomos)
grafo = BloqadeMIS.unit_disk_graph(atomos, 7.5) #Genera el grafo del problema.
independent_set_problem = IndependentSet(grafo) #Crea el problema.
mis_size_and_counting = GenericTensorNetworks.solve(GenericTensorNetwork(independent_set_problem), CountingMax())[ ]
#Esta última línea resuelve de manera análitica el problema y determina el tamaño del MIS y el número de soluciones que hay.

(30.0, 129364.0)ₜ

Una vez definido y analizado el problema, comenzamos a preparar las funciones para abordar el problema mediante el paquete Bloqade. El primer paso es preparar el hamiltoniano y los valores de la frecuencia de Rabi y del detuning.

In [4]:
Δ = 2π * 10 #Valor de detuning.
Ω = 2π * 4 #Valor de la frecuencia de Rabi.
h = rydberg_h(atomos; Δ, Ω) #Hamiltoniano.
h_qmc = rydberg_qmc(h); #Hamiltoniano apto para QMC (formalismo SSE).

Cabe destacar que aquí damos un valor fijo a la frecuencia de Rabi porque realizamos QMC durante Ω está fija en ese valor, de ahí la diferencia de pasar de una función a un valor fijo. Veremos adelante que sí que tomaremos distintos valores para el detuning a la hora de hacer QMC. 

Seguimos definiendo el número de pasos para la equilibración del sistema, así como el número de pasos para MC:

In [5]:
EQ_MCS = 500; #Equilibration Monte Carlo steps.
MCS = 10000; #Monte Carlo steps.

Definimos ahora el orden de la expansión SSE con los objetos necesarios para analizar el sistema a lo largo de la evolución:

In [6]:
M = 5000;
ts = BinaryThermalState(h_qmc, M); #Guarda la información de la configuración SSE.
d = Diagnostics(); #Para analizar más en detalle la configuración SSE.

Definimos el inverso de la temperatura lo suficientemente grande para que el sistema alcance el estado fundamental y inicializamos el generador de números pseudoaleatorios.

In [7]:
β = 5;
rng = MersenneTwister(13);

## Quantum Monte Carlo

Empezamos la simulación por Monte Carlo del sistema.

In [ ]:
# 1. Ajustes iniciales para 100 átomos
EQ_MCS = 500   # Le damos un poco más de tiempo para equilibrar
MCS = 10000    # Reducimos los pasos a 10,000 para probar (suele ser suficiente)

M = 8000       # ¡CRUCIAL! Aumentamos el tamaño de la expansión SSE
ts = BinaryThermalState(h_qmc, M)
d = Diagnostics()

# 2. Bloque 'let' para rendimiento extremo en Julia
@time let rng=rng, ts=ts, h_qmc=h_qmc, β=β, d=d, MCS=MCS, n_sites=n_sites, EQ_MCS=EQ_MCS
    
    println("Iniciando equilibración...")
    for _ in 1:EQ_MCS
        mc_step_beta!(rng, ts, h_qmc, β, d, eq = true)
    end
    
    println("Iniciando muestreo...")
    global occs = zeros(Float64, MCS, n_sites)
    
    for i in 1:MCS
        mc_step_beta!(rng, ts, h_qmc, β, d, eq = false) do lsize, ts_step, h_step
            SSE_slice = sample(h_step, ts_step, 1)
            
            # Escribimos directo en la matriz sin crear arreglos intermedios
            @inbounds for j in 1:n_sites
                occs[i, j] = SSE_slice[j] ? 1.0 : 0.0
            end
        end
    end
end

# 3. Cálculo final (fuera del bloque let)
densidades_QMC = [mean(occs[:, jj]) for jj in 1:n_sites]
println("¡Simulación terminada!")

Iniciando equilibración...


Dibujamos el resultado:

In [ ]:
# 1. Corregido: Iterar directamente sobre 'atoms' para obtener las coordenadas
x_coords = [pos[1] for pos in atomos]
y_coords = [pos[2] for pos in atomos]

# 2. Configurar la figura y el eje
fig = Figure(size = (750, 600))
ax = Axis(fig[1, 1];
    xlabel = "Posición x (μm)",
    ylabel = "Posición y (μm)",
    title = "Densidades de Ocupación QMC (Red Cuadrada con dropout)",
    aspect = DataAspect()) # Mantiene la proporción física de la red cuadrada

# 3. Graficar los datos usando el vector de densidades calculado
plt = scatter!(ax, x_coords, y_coords;
    color = densidades_QMC, 
    markersize = 20, 
    colormap = :viridis)

# 4. Añadir la barra de color
Colorbar(fig[1, 2], plt; label = "Densidad media")

fig

In [ ]:
# 1. Binarizar las densidades usando un umbral
umbral = 0.5
estado_candidato = densidades_QMC .> umbral
n_excitados = sum(estado_candidato)

println("--- ANÁLISIS DE RESULTADOS ---")
println("Átomos en el conjunto (QMC): ", n_excitados)
# mis_size_and_counting suele ser un log-polinomio, extraemos la potencia máxima (el tamaño del MIS)
println("Solución analítica (Tensor Networks): ", mis_size_and_counting) 

# 2. Comprobar violaciones del bloqueo de Rydberg
violaciones = 0
# Iteramos sobre las aristas del grafo (las conexiones donde los átomos están a menos de 7.5 μm)
for arista in edges(grafo)
    # Si ambos átomos conectados por la arista están "excitados", hay una violación
    if estado_candidato[src(arista)] && estado_candidato[dst(arista)]
        violaciones += 1
    end
end

if violaciones == 0
    println("✅ ¡Es un Independent Set válido! Cero violaciones de bloqueo.")
    if n_excitados == maximum(mis_size_and_counting).size # o el atributo que guarde el tamaño exacto
        println("🏆 ¡Y es un MAXIMUM Independent Set perfecto!")
    else
        println("⚠️ Sin embargo, es un óptimo local. Aún faltan átomos para llegar al máximo global.")
    end
else
    println("❌ Se encontraron $violaciones violaciones del bloqueo. No es un Independent Set estricto.")
end